# SpeakUp — Gemma 4 Fine-tuning with Unsloth

**Kaggle Gemma 4 Good Hackathon — Unsloth Special Track ($10,000)**

> This notebook fine-tunes Gemma 4 on 61 expert-crafted AAC (Augmentative & Alternative Communication) scenarios using Unsloth LoRA. The fine-tuned model is then exported to GGUF format for use with Ollama in the SpeakUp app.

**Recommended runtime: Colab T4 for E2B, A100 for E4B/31B**

---

## What fine-tuning does
```
Base Gemma 4 (general purpose)
         ↓  LoRA adapters trained on AAC data
SpeakUp Gemma 4 (specialized for AAC intent prediction)
         ↓  Converted to GGUF
Loaded into Ollama on your laptop → used by SpeakUp app
```

**Before fine-tuning:** Model has no knowledge of AAC, may produce invalid JSON, generic responses
**After fine-tuning:** Model reliably outputs structured JSON, understands AAC signals, accurately infers intent

## Step 1: Install Unsloth and dependencies

Unsloth makes fine-tuning 2-5x faster with 70% less memory. Required for the Unsloth special track prize.

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

## Step 2: Load Gemma 4

We use Unsloth's 4-bit quantized version. This fits in 16GB VRAM and trains 2x faster than full precision.

> **Note:** The notebook is configured for Gemma 4 only. If Unsloth publishes a different Gemma 4 checkpoint name, update `MODEL_NAME` to that Gemma 4 checkpoint before running.

In [ ]:
from unsloth import FastLanguageModel
import torch

# === CONFIGURATION ===
# Update this when Gemma 4 is on Unsloth Hub:
MODEL_NAME = 'unsloth/gemma-4-E2B-it-unsloth-bnb-4bit'
# MODEL_NAME = 'unsloth/gemma-4-E4B-it-unsloth-bnb-4bit'  # A100/stronger GPU

MAX_SEQ_LENGTH = 1024
LORA_RANK = 8         # T4-safe. Use 16 or 32 on A100.
LORA_ALPHA = 16       # Usually 2x the rank
NUM_EPOCHS = 4        # More epochs = better fit on small dataset
BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4
OUTPUT_DIR = './speakup-gemma4-lora'

print(f'Loading {MODEL_NAME}...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
print('Model loaded successfully!')

## Step 3: Add LoRA Adapters

**What is LoRA?** Low-Rank Adaptation adds small trainable weight matrices to the model. Instead of updating all 9 billion parameters (which would require 80GB+ VRAM), LoRA only trains ~20 million parameters in the adapter layers.

```
Frozen base weights (9B params, untouched)
    + LoRA adapters (20M params, trained on AAC data)
    = Specialized SpeakUp model
```

This is what enables fine-tuning on a single A100 GPU.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=True,   # Rank-stabilized LoRA — better results
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} ({100*trainable/total:.2f}% of {total:,} total)')

## Step 4: Load AAC Training Dataset

Our dataset has 61 expert-crafted AAC scenarios covering:
- Water/drink requests (5 examples)
- Food/hunger (5)
- Sensory overwhelm (5)
- Pain/medical urgency (4)
- Emotional needs (5)
- Activities/preferences (4)
- Social/people (3)
- Bathroom (2)
- Ambiguous signals (4)
- Multi-signal complex (4)
- And more...

Each example teaches the model:
1. How to read multimodal AAC signals
2. How to weight confirmed memory patterns
3. How to produce accurate JSON structure
4. How to calibrate confidence scores correctly
5. How to detect urgency and emotion

In [ ]:
import json
from datasets import Dataset

# Load the training data (upload aac_training.jsonl from the finetune/dataset/ folder)
# OR paste the path to your uploaded file
DATASET_PATH = '/content/aac_training.jsonl'

# If you haven't uploaded it yet, create a small inline version:
import os
if not os.path.exists(DATASET_PATH):
    print('Dataset not found at', DATASET_PATH)
    print('Please upload finetune/dataset/aac_training.jsonl to Colab')
    print('Or download it from your SpeakUp repo')
else:
    raw_data = []
    with open(DATASET_PATH) as f:
        for line in f:
            raw_data.append(json.loads(line.strip()))
    print(f'Loaded {len(raw_data)} training examples')

def format_for_gemma(example):
    msgs = example['messages']
    text = ''
    for msg in msgs:
        if msg['role'] == 'system':
            # Gemma chat format
            text += f"<start_of_turn>user\n[System]: {msg['content']}\n"
        elif msg['role'] == 'user':
            text += f"{msg['content']}<end_of_turn>\n<start_of_turn>model\n"
        elif msg['role'] == 'assistant':
            text += f"{msg['content']}<end_of_turn>\n"
    return {'text': text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_for_gemma, remove_columns=dataset.column_names)
print(f'Dataset formatted. Sample:')
print(dataset[0]['text'][:400])

## Step 5: Fine-tune!

This is the core training step. Expected time:
- A100 (80GB): ~8 minutes
- A100 (40GB): ~12 minutes  
- T4: ~45 minutes (may OOM — reduce batch size)

Watch the **loss** decrease. Good target: below 0.3 by end of epoch 4.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,  # Pack multiple short examples into one sequence — faster
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        warmup_ratio=0.1,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        output_dir=OUTPUT_DIR,
        report_to='none',
        save_strategy='epoch',
    ),
)

print('Starting fine-tuning...')
print('Watch the loss — it should decrease each epoch')
trainer_stats = trainer.train()
print(f'Training complete! Final loss: {trainer_stats.training_loss:.4f}')

## Step 6: Test the Fine-tuned Model

Let's verify the model produces correct JSON output for AAC scenarios.

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM = 'You are SpeakUp AI intent engine for minimally speaking and non-speaking communicators. Respond ONLY with valid JSON.'

test_cases = [
    'Communicator: points at cup, soft mmm sound. Time: afternoon. Memory: soft mmm plus cup = water confirmed 4x.',
    'Communicator: covers ears, rocks body. Time: during TV. Memory: covers ears = too loud confirmed 6x.',
    'Communicator: selected Pain card, holding stomach, crying. Time: after meal. No memory.',
]

import json as json_lib

print('=== Fine-tuned Model Inference Tests ===\n')
successes = 0
for test in test_cases:
    prompt = f'<start_of_turn>user\n[System]: {SYSTEM}\n{test}<end_of_turn>\n<start_of_turn>model\n'
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.15,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    result = result.split('<end_of_turn>')[0].strip()
    
    print(f'Input: {test[:80]}...')
    try:
        parsed = json_lib.loads(result)
        print(f'Intent: {parsed.get("intent")}')
        print(f'Confidence: {parsed.get("confidence")}')
        print(f'Urgency: {parsed.get("urgency")}')
        print(f'JSON valid: YES')
        successes += 1
    except:
        print(f'JSON valid: NO — raw output: {result[:200]}')
    print()

print(f'JSON success rate: {successes}/{len(test_cases)} ({100*successes//len(test_cases)}%)')

## Step 7: Save the LoRA Adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to {OUTPUT_DIR}')

import os
files = os.listdir(OUTPUT_DIR)
print(f'Files saved: {files}')

## Step 8: Export to GGUF for Ollama

This converts the fine-tuned model to GGUF format (Q4_K_M quantization) so it can be loaded by Ollama on your laptop.

**GGUF = the format Ollama uses to run models locally**

In [ ]:
print('Merging LoRA weights and exporting to GGUF...')
print('This takes about 3-5 minutes...')

model.save_pretrained_gguf(
    'speakup-gemma4',
    tokenizer,
    quantization_method='q4_k_m'  # Good balance of quality vs size
)

import os
gguf_file = 'speakup-gemma4-unsloth.Q4_K_M.gguf'
if os.path.exists(gguf_file):
    size = os.path.getsize(gguf_file) / 1e9
    print(f'GGUF saved: {gguf_file} ({size:.1f} GB)')
    print('Download this file — you need it for Step 9')
else:
    # Unsloth may name it differently
    for f in os.listdir('.'):
        if '.gguf' in f:
            size = os.path.getsize(f) / 1e9
            print(f'Found GGUF: {f} ({size:.1f} GB)')

## Step 9: Download the GGUF file

Download `speakup-gemma4-unsloth.Q4_K_M.gguf` from Colab to your computer.

**Then on your laptop:**

```bash
# 1. Create Modelfile
cat > Modelfile << 'EOF'
FROM ./speakup-gemma4-unsloth.Q4_K_M.gguf

SYSTEM You are SpeakUp AI intent engine for minimally speaking and non-speaking communicators. Respond ONLY with valid JSON.

PARAMETER temperature 0.15
PARAMETER top_p 0.9
PARAMETER num_predict 512
EOF

# 2. Create the Ollama model
ollama create speakup-gemma4 -f Modelfile

# 3. Test it
ollama run speakup-gemma4 "Communicator: points at cup. Time: afternoon. Predict intent."

# 4. Update SpeakUp backend
# In backend/.env: OLLAMA_MODEL=speakup-gemma4

# 5. Restart backend
cd backend && source venv/bin/activate && uvicorn main:app --reload
```

In [ ]:
# Download the GGUF file from Colab
from google.colab import files
import os

# Find the GGUF file
gguf_files = [f for f in os.listdir('.') if f.endswith('.gguf')]
if gguf_files:
    gguf_path = gguf_files[0]
    print(f'Downloading {gguf_path}...')
    print('NOTE: This is 4-6GB — may take a few minutes to download')
    files.download(gguf_path)
else:
    print('No GGUF file found. Check previous step.')

## Appendix: Benchmarking

Run this to measure improvement from fine-tuning.

In [ ]:
# Benchmark: JSON reliability and accuracy
import json as json_lib
import time

benchmark_cases = [
    ('water', 'Communicator: points at cup, soft mmm. Time: afternoon. Memory: mmm plus cup = water confirmed 4x.'),
    ('pain-urgent', 'Communicator: selected Pain card, holding stomach, crying. Time: after meal.'),
    ('overwhelm', 'Communicator: covers ears, rocks body. Memory: covers ears = too loud confirmed 8x.'),
    ('ambiguous', 'Communicator: new behavior taps head. No memory.'),
    ('happy', 'Communicator: laughing, Happy card, bouncing. Time: during favorite video.'),
]

results = []
for label, test in benchmark_cases:
    prompt = f'<start_of_turn>user\n[System]: You are SpeakUp AI. JSON only.\n{test}<end_of_turn>\n<start_of_turn>model\n'
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.15, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - start
    raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).split('<end_of_turn>')[0].strip()
    try:
        parsed = json_lib.loads(raw)
        valid = True
        conf = parsed.get('confidence', 0)
        urgency = parsed.get('urgency', '?')
        if label == 'pain-urgent' and urgency != 'high':
            urgency_correct = False
        else:
            urgency_correct = True
    except:
        valid = False; conf = 0; urgency = '?'; urgency_correct = False
    results.append({'label': label, 'json_valid': valid, 'confidence': conf, 'time': elapsed, 'urgency_ok': urgency_correct})
    print(f'{label}: JSON={valid}, conf={conf:.2f}, urgency_ok={urgency_correct}, time={elapsed:.1f}s')

json_rate = sum(1 for r in results if r['json_valid']) / len(results)
urgency_rate = sum(1 for r in results if r['urgency_ok']) / len(results)
avg_time = sum(r['time'] for r in results) / len(results)
print(f'\nJSON success: {json_rate:.0%} | Urgency accuracy: {urgency_rate:.0%} | Avg time: {avg_time:.1f}s')